In [9]:
import os
import sys
import pandas as pd

# 경로 설정
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(BASE_DIR, "data")
sys.path.append(BASE_DIR)

# 파일 로드
gh_path = os.path.join(DATA_DIR, "code_review_gh", "data", "code_review_gh_2023.parquet")
context_path = os.path.join(DATA_DIR, "contextual_code_review", "cleaned_data.json")
reviewer_path = os.path.join(DATA_DIR, "codereviewer", "generation", "gen-train.jsonl")

print("⏳ 원본 데이터 로딩 중...")
df_gh = pd.read_parquet(gh_path)
df_context = pd.read_json(context_path)
df_reviewer = pd.read_json(reviewer_path, lines=True)
print("✅ 메모리 로드 완료!")

⏳ 원본 데이터 로딩 중...
✅ 메모리 로드 완료!


In [18]:
import importlib
import utils.adapter

# adapter.py 변경사항 강제 새로고침
importlib.reload(utils.adapter)
from utils.adapter import adapt_code_review_gh, adapt_contextual_code_review, adapt_codereviewer

print("⏳ 정제 규칙 적용 및 변환 처리 시작...\n")

# 1. code_review_gh
df_gh_adapted = adapt_code_review_gh(df_gh)
df_gh_adapted['dataset_source'] = 'code_review_gh'
print(f"1️⃣ code_review_gh: {len(df_gh):,}개 ➡️ {len(df_gh_adapted):,}개 (약 {len(df_gh) - len(df_gh_adapted):,}개 필터링됨)")

# 2. contextual_code_review
df_context_adapted = adapt_contextual_code_review(df_context)
df_context_adapted['dataset_source'] = 'contextual_code_review'
print(f"2️⃣ contextual_code_review: {len(df_context):,}개 ➡️ {len(df_context_adapted):,}개 (약 {len(df_context) - len(df_context_adapted):,}개 필터링됨)")

# 3. codereviewer
df_reviewer_adapted = adapt_codereviewer(df_reviewer)
df_reviewer_adapted['dataset_source'] = 'codereviewer'
print(f"3️⃣ codereviewer: {len(df_reviewer):,}개 ➡️ {len(df_reviewer_adapted):,}개 (약 {len(df_reviewer) - len(df_reviewer_adapted):,}개 필터링됨)")

# Unified Dataset 병합
unified_df = pd.concat([df_gh_adapted, df_context_adapted, df_reviewer_adapted], ignore_index=True)

print("\n" + "=" * 50)
print(f"🎉 최종 정제 완료된 통합 데이터셋: 총 {len(unified_df):,}개")
print("=" * 50)

⏳ 정제 규칙 적용 및 변환 처리 시작...

1️⃣ code_review_gh: 747,555개 ➡️ 630,637개 (약 116,918개 필터링됨)
2️⃣ contextual_code_review: 61,935개 ➡️ 55,276개 (약 6,659개 필터링됨)
3️⃣ codereviewer: 117,739개 ➡️ 117,739개 (약 0개 필터링됨)

🎉 최종 정제 완료된 통합 데이터셋: 총 803,652개


In [19]:
print("=" * 50)
print(f"🎉 통합 데이터셋 총 레코드 수: {len(unified_df):,}개")
print("=" * 50)
print(unified_df.groupby('dataset_source').size())
print("\n--- 통합 데이터 미리보기 ---")
unified_df[['dataset_source', 'has_issue', 'review_comment']].head(5)

🎉 통합 데이터셋 총 레코드 수: 803,652개
dataset_source
code_review_gh            630637
codereviewer              117739
contextual_code_review     55276
dtype: int64

--- 통합 데이터 미리보기 ---


,dataset_source,has_issue,review_comment
0,code_review_gh,True,Can we add gpgme as an optional dependencies. ...
1,code_review_gh,True,Can we specify lua version to luajit because t...
2,code_review_gh,True,"can we use ubuntu-22.04 here, a LTS is a bette..."
3,code_review_gh,True,Can we add GPGme as optional dependencies?
4,code_review_gh,True,"Just let it be like this, I have plan to repla..."


In [22]:
# 정제 및 변환이 완료된 *_adapted 데이터셋 분석
inspect_dataset("code_review_gh (정제후)", df_gh_adapted, comment_col="review_comment", code_col="source_code")

inspect_dataset("contextual_code_review (정제후)", df_context_adapted, comment_col="review_comment", code_col="source_code")

inspect_dataset("codereviewer (정제후)", df_reviewer_adapted, comment_col="review_comment", code_col="source_code")

📊 [code_review_gh (정제후)] 데이터셋 분석 결과
• 전체 레코드 수: 630,637개
• 결측치(Null) -> 코멘트: 0개 / 코드: 0개
• 코멘트 길이 -> 평균: 207.7자 / 중앙값: 102자 / 최소: 15자 / 최대: 53234자
• 15자 미만 짧은 코멘트: 0개 (0.00%)


📊 [contextual_code_review (정제후)] 데이터셋 분석 결과
• 전체 레코드 수: 55,276개
• 결측치(Null) -> 코멘트: 0개 / 코드: 0개
• 코멘트 길이 -> 평균: 164.9자 / 중앙값: 95자 / 최소: 15자 / 최대: 44962자
• 15자 미만 짧은 코멘트: 0개 (0.00%)


📊 [codereviewer (정제후)] 데이터셋 분석 결과
• 전체 레코드 수: 117,739개
• 결측치(Null) -> 코멘트: 0개 / 코드: 0개
• 코멘트 길이 -> 평균: 115.4자 / 중앙값: 86자 / 최소: 6자 / 최대: 3421자
• 15자 미만 짧은 코멘트: 308개 (0.26%)

  [자주 나오는 짧은 코멘트 Top 5]
    - 'Is this used?': 14회
    - 'rm empty line': 4회
    - 'Why remove it?': 4회
    - 'why remove it?': 4회
    - 'he -> The': 4회




In [23]:
import numpy as np
import pandas as pd

# 1. 길이 계산 (Character Count)
unified_df['code_char_len'] = unified_df['source_code'].astype(str).str.len()
unified_df['diff_char_len'] = unified_df['pr_diff'].astype(str).str.len()
unified_df['comment_char_len'] = unified_df['review_comment'].astype(str).str.len()

print("==================================================")
print("📏 통합 데이터셋 길이 분포 통계 (Character Count)")
print("==================================================")

stats_df = pd.DataFrame({
    'Source Code 길이': unified_df['code_char_len'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]),
    'PR Diff 길이': unified_df['diff_char_len'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]),
    'Review Comment 길이': unified_df['comment_char_len'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
})

print(stats_df.round(1))

# 2. 대략적인 Token 수 추정 (코드/영문 기준 1 토큰 ≒ 약 3.5~4자)
print("\n" + "=" * 50)
print("💡 90% ~ 95% 커버리지를 위한 추정 토큰 수")
print("=" * 50)
p90_code = np.percentile(unified_df['code_char_len'], 90)
p95_code = np.percentile(unified_df['code_char_len'], 95)

print(f"• Source Code 90% 퍼센타일: {p90_code:,.0f} 자 (약 {p90_code/3.5:,.0f} 토큰)")
print(f"• Source Code 95% 퍼센타일: {p95_code:,.0f} 자 (약 {p95_code/3.5:,.0f} 토큰)")

📏 통합 데이터셋 길이 분포 통계 (Character Count)
       Source Code 길이  PR Diff 길이  Review Comment 길이
count        803652.0    803652.0           803652.0
mean           5157.2       846.3              191.2
std           33156.7      1092.0              353.6
min               0.0         0.0                6.0
50%             732.0       523.0               99.0
75%            1963.0      1057.0              188.0
90%            6746.0      2081.0              374.0
95%           21566.4      2803.0              648.0
99%           88399.4      3727.0             1729.0
max        14065259.0    214629.0            53234.0

💡 90% ~ 95% 커버리지를 위한 추정 토큰 수
• Source Code 90% 퍼센타일: 6,746 자 (약 1,927 토큰)
• Source Code 95% 퍼센타일: 21,566 자 (약 6,162 토큰)


In [26]:
import os

# 프로젝트 루트 기준 data 디렉토리 경로 지정
DATA_DIR = os.path.join("..", "data")
os.makedirs(DATA_DIR, exist_ok=True)  # 폴더 없으면 생성

unified_path = os.path.join(DATA_DIR, "unified_data_cleaned.parquet")

# unified_df를 파일로 저장
unified_df.to_parquet(unified_path, index=False)

print(f"💾 고품질 통합 데이터셋 저장 완료!")
print(f"• 저장 경로: {unified_path}")
print(f"• 데이터 건수: {len(unified_df):,} 개")

💾 고품질 통합 데이터셋 저장 완료!
• 저장 경로: ..\data\unified_data_cleaned.parquet
• 데이터 건수: 803,652 개


In [29]:
import os
import sys
import pandas as pd
import importlib
from dotenv import load_dotenv

# 1. 프로젝트 Root 디렉토리를 파이썬 모듈 검색 경로에 추가
sys.path.append(os.path.abspath(".."))

# 2. 프로젝트 루트의 .env 파일 로드
load_dotenv(os.path.join("..", ".env"))

# 3. scripts 및 core 모듈 불러오기
import scripts.index_to_pg as indexer
importlib.reload(indexer)

from core.db import engine  # core/db.py

# 4. 정제된 파켓 데이터 로드
DATA_DIR = os.path.join("..", "data")
unified_path = os.path.join(DATA_DIR, "unified_data_cleaned.parquet")

print("⏳ 정제된 파켓 데이터 로딩 중...")
df_clean = pd.read_parquet(unified_path)

# 5. POC 테스트용 1만 건 샘플링
sample_size = 10000
sample_df = df_clean.sample(n=sample_size, random_state=42).reset_index(drop=True)
print(f"🎯 임베딩 및 DB 인덱싱 테스트용 {sample_size:,}건 준비 완료!")

# 6. DB 자동 생성, HNSW 인덱싱 & OpenAI 임베딩 저장 실행
# scripts 모듈 새로고침 후 재실행
importlib.reload(indexer)

# batch_size=30, delay_seconds=1.5 로 안전하게 재실행
indexer.embed_and_insert(sample_df, engine, batch_size=30, delay_seconds=1.5)

⏳ 정제된 파켓 데이터 로딩 중...
🎯 임베딩 및 DB 인덱싱 테스트용 10,000건 준비 완료!
✅ PostgreSQL 테이블 및 HNSW vector 인덱스 준비 완료!
🚀 총 10,000건 데이터 임베딩 및 DB 저장 시작 (Batch Size: 30)...
  - [30/10,000] 건 저장 완료
  - [60/10,000] 건 저장 완료
  - [90/10,000] 건 저장 완료
  - [120/10,000] 건 저장 완료
  - [150/10,000] 건 저장 완료
  - [180/10,000] 건 저장 완료
  - [210/10,000] 건 저장 완료
  - [240/10,000] 건 저장 완료
  - [270/10,000] 건 저장 완료
  - [300/10,000] 건 저장 완료
  - [330/10,000] 건 저장 완료
  - [360/10,000] 건 저장 완료
  - [390/10,000] 건 저장 완료
  - [420/10,000] 건 저장 완료
  - [450/10,000] 건 저장 완료
  - [480/10,000] 건 저장 완료
  - [510/10,000] 건 저장 완료
  - [540/10,000] 건 저장 완료
  - [570/10,000] 건 저장 완료
  - [600/10,000] 건 저장 완료
  - [630/10,000] 건 저장 완료
  - [660/10,000] 건 저장 완료
  - [690/10,000] 건 저장 완료
  - [720/10,000] 건 저장 완료
  - [750/10,000] 건 저장 완료
  - [780/10,000] 건 저장 완료
  - [810/10,000] 건 저장 완료
  - [840/10,000] 건 저장 완료
  - [870/10,000] 건 저장 완료
  - [900/10,000] 건 저장 완료
  - [930/10,000] 건 저장 완료
  - [960/10,000] 건 저장 완료
  - [990/10,000] 건 저장 완료
  - [1,020/10,000] 건 저장 완료
  

In [ ]:
import sys
import os
from sqlalchemy import text
from openai import OpenAI
from dotenv import load_dotenv

# 1. 환경 변수 및 DB 엔진 로드
sys.path.append(os.path.abspath(".."))
load_dotenv(os.path.join("..", ".env"))

from core.db import engine

client = OpenAI()

# 2. 유사도 검색 함수
def search_similar_code_reviews(query_text: str, top_k: int = 3):
    # 입력 질의문 임베딩
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[query_text]
    )
    query_vector = response.data[0].embedding

    # ::vector 캐스팅 대신 CAST(... AS vector) 문법으로 구문 오류 수정
    search_sql = text("""
        SELECT 
            dataset_source,
            source_code,
            pr_diff,
            review_comment,
            has_issue,
            1 - (embedding <=> CAST(:query_vector AS vector)) AS similarity
        FROM code_reviews
        ORDER BY embedding <=> CAST(:query_vector AS vector) ASC
        LIMIT :top_k;
    """)

    with engine.connect() as conn:
        results = conn.execute(
            search_sql, 
            {"query_vector": str(query_vector), "top_k": top_k}
        ).fetchall()

    print("=" * 65)
    print(f"🔍 입력 질의문(Query): '{query_text}'")
    print("=" * 65)

    for idx, row in enumerate(results, 1):
        print(f"\n🏆 [TOP {idx}] 유사도 점수(Similarity): {row.similarity:.4f}")
        print(f"📌 출처: {row.dataset_source} | 이슈 여부: {row.has_issue}")
        print(f"💬 리뷰 코멘트:\n   └─ {row.review_comment}")
        print(f"📝 관련 PR Diff (미리보기):")
        diff_preview = str(row.pr_diff)[:200].replace('\n', '\n   ')
        print(f"   {diff_preview}...")
        print("-" * 65)

# 3. 테스트 실행!
search_similar_code_reviews("Check if list or array is null or empty before iterating", top_k=3)

ProgrammingError: (psycopg2.errors.SyntaxError) 오류:  구문 오류, ":" 부근
LINE 8:             1 - (embedding <=> :query_vector::vector) AS sim...
                                       ^

[SQL: 
        SELECT 
            dataset_source,
            source_code,
            pr_diff,
            review_comment,
            has_issue,
            1 - (embedding <=> :query_vector::vector) AS similarity
        FROM code_reviews
        ORDER BY embedding <=> :query_vector::vector ASC
        LIMIT %(top_k)s;
    ]
[parameters: {'top_k': 3}]
(Background on this error at: https://sqlalche.me/e/20/f405)